In [4]:
pip install aiohttp aiofiles pymongo python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 2.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 2.5 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [40]:
import asyncio
import aiohttp
import aiofiles
import os
import sys
import re
from datetime import datetime

# Point at your Utilities folder
sys.path.insert(0, os.path.join(os.getcwd(), "..", "Utilities"))
import config
import db

# ==============================================================
# 1. CONFIGURATION
# ==============================================================
IS_WINDOWS = os.name == 'nt'

if IS_WINDOWS:
    SAVE_ROOT = r"T:\Full Card Database\New Cards"
    CHECK_ROOT = r"T:\Full Card Database\Folder Database"
else:
    SAVE_ROOT = os.path.expanduser("/storage/Tera/Full Card Database/New Cards")
    CHECK_ROOT = os.path.expanduser("/storage/Tera/Full Card Database/Folder Database")

MAX_CONCURRENT_DOWNLOADS = 30
RUN_MODE = "all"          # "menu" | "all" | int (1-based game)
SHOW_PROGRESS = True

# HD settings
HD_SIZE = "1000x1000"     # the HD resolution token to request

# Mongo accessors
master_coll = lambda: db.get_db()[config.TCG_MASTER_COLLECTION]
skipped_coll = lambda: db.get_db()[config.SKIPPED_IMAGES_COLLECTION]

# Card collection for harvested card records
CARD_COLLECTION = "tcg_folder_harvester"
card_coll = lambda: db.get_db()[CARD_COLLECTION]

# Categories with no usable groups/products (from tcgcsv)
EMPTY_CATEGORIES = {9, 10, 12, 14, 21, 55, 69, 70}

new_dl_count = 0
counter_lock = asyncio.Lock()


# ==============================================================
# 2. CATEGORY MAPPING
# ==============================================================
async def load_category_map(session):
    url = "https://tcgcsv.com/tcgplayer/categories"
    try:
        async with session.get(url, timeout=15) as resp:
            if resp.status == 200:
                data = await resp.json(content_type=None)
                categories = data.get("results", [])
                lookup = {}
                for cat in categories:
                    cid = cat.get("categoryId")
                    if cid is None:
                        continue
                    names = {
                        str(cat.get("displayName", "")).strip().lower(),
                        str(cat.get("name", "")).strip().lower(),
                        str(cat.get("seoCategoryName", "")).strip().lower(),
                    }
                    for n in names:
                        if n:
                            lookup.setdefault(n, cid)
                return lookup, categories
    except Exception as e:
        print(f"WARNING: could not load categories: {e}")
    return {}, []


def resolve_category_id(row, category_map, categories):
    stored = row.get("site_id")
    if stored is not None and str(stored).strip():
        try:
            sid = int(stored)
            for cat in categories:
                if cat.get("categoryId") == sid:
                    return sid
        except (ValueError, TypeError):
            pass
    name = str(row.get("tcg_display_name", "")).strip().lower()
    if name in category_map:
        return category_map[name]
    for cat in categories:
        haystack = " ".join([
            str(cat.get("displayName", "")).lower(),
            str(cat.get("name", "")).lower(),
            str(cat.get("seoCategoryName", "")).lower(),
        ])
        if name and name in haystack:
            return cat.get("categoryId")
    return None


# ==============================================================
# 3. MENU
# ==============================================================
def show_menu(rows, global_total):
    print(f"\nENGLISH TCG HARVESTER (MONGODB / JUPYTER EDITION)")
    print(f"Total skipped records: {global_total:,}")
    print("-" * 100)
    print(f"{'#':<4} {'TCG CATEGORY':<25} | {'SITE ID':<10} | {'LAST RUN':<18} | FOLDER")
    print("-" * 100)
    for i, r in enumerate(rows, 1):
        last = r.get('last_run') or "Never"
        print(f"{i:<4} {r.get('tcg_display_name',''):<25} | {str(r.get('site_id','')):<10} | {str(last):<18} | {r.get('folder_name','')}")
    print("-" * 100)


# ==============================================================
# 4. ASYNC CORE LOGIC
# ==============================================================
async def fetch_json(session, url):
    try:
        async with session.get(url, timeout=15) as resp:
            if resp.status == 200:
                data = await resp.json(content_type=None)
                if isinstance(data, dict):
                    return data.get('results', [])
                if isinstance(data, list):
                    return data
    except Exception:
        pass
    return []


def make_hd_url(std_url):
    """
    Build the HD URL from the standard one by replacing the
    size token (e.g. _200x200) with the HD size.
    e.g. .../26056_in_200x200.jpg -> .../26056_in_1000x1000.jpg
    """
    if not std_url:
        return None
    hd = re.sub(r'_\d+x\d+(?=\.[a-zA-Z0-9]+$)', f'_{HD_SIZE}', std_url)
    if hd == std_url:
        hd = re.sub(r'(\.[a-zA-Z0-9]+)$', f'_{HD_SIZE}\\1', std_url)
    return hd


def clean_image_name(product_id):
    """Drop the _in prefix and the size token -> just the productId + .jpg"""
    return f"{product_id}.jpg"


async def pick_best_url(session, hd_url, std_url):
    """Return (url, is_hd). Try HD first, fall back to standard."""
    if hd_url:
        try:
            async with session.get(hd_url, timeout=15) as resp:
                if resp.status == 200:
                    return hd_url, True
        except Exception:
            pass
    return std_url, False


async def download_image(session, product, set_folder, skipped_ids, existing_filenames, semaphore):
    global new_dl_count
    async with semaphore:
        product_id = product.get('productId')
        std_url = product.get('imageUrl')
        if not product_id or not std_url:
            return

        clean_name = clean_image_name(product_id)
        if clean_name in skipped_ids or clean_name in existing_filenames:
            return

        hd_url = make_hd_url(std_url)
        url, is_hd = await pick_best_url(session, hd_url, std_url)

        image_path = os.path.join(set_folder, clean_name)
        try:
            async with session.get(url, timeout=20) as response:
                if response.status == 200:
                    os.makedirs(set_folder, exist_ok=True)
                    data = await response.read()
                    async with aiofiles.open(image_path, 'wb') as f:
                        await f.write(data)

                    # Delete old non-clean variants of this product
                    for old in list(os.listdir(set_folder)):
                        if old.startswith(f"{product_id}_") and old != clean_name:
                            try:
                                os.remove(os.path.join(set_folder, old))
                            except OSError:
                                pass

                    async with counter_lock:
                        new_dl_count += 1

                    # Record the card in Mongo (matches tcg_folder_harvester schema)
                    card_coll().update_one(
                        {"product_id": str(product_id)},
                        {"$set": {
                            "product_id": str(product_id),
                            "game": product.get("game_name"),
                            "group_name": product.get("set_name"),
                            "filename": clean_name,
                            "path": image_path,
                            "extension": os.path.splitext(clean_name)[1],
                            "HD": 1 if is_hd else 0,
                            "record_type": "image",
                            "status": "available",
                            "confirmed": True,
                            "file_size": len(data),
                            "modified_time": datetime.now().isoformat(),
                            "updated_at": datetime.now().isoformat(),
                            "last_seen": datetime.now().isoformat(),
                        }},
                        upsert=True,
                    )
        except Exception:
            pass


async def process_tcg(session, row, category_map, categories):
    global new_dl_count
    new_dl_count = 0
    name = row['tcg_display_name']
    folder_name = row['folder_name']

    print(f"\nPROCESSING: {name.upper()} ({folder_name})")

    sid = resolve_category_id(row, category_map, categories)
    if sid is None:
        print(f"  !! Could not resolve a categoryId for '{name}'. Skipping.")
        return
    if sid in EMPTY_CATEGORIES:
        print(f"  !! Category {sid} is a known empty category. Skipping.")
        return
    print(f"  (categoryId = {sid})")

    skipped_ids = {
        str(d['image_name'])
        for d in skipped_coll().find({"language": "english", "game_name": name}, {"image_name": 1})
    }

    save_dir = os.path.join(SAVE_ROOT, folder_name)
    check_dir = os.path.join(CHECK_ROOT, folder_name)

    existing_files = set()
    for path in [save_dir, check_dir]:
        if os.path.exists(path):
            for root, _, files in os.walk(path):
                existing_files.update(files)

    semaphore = asyncio.Semaphore(MAX_CONCURRENT_DOWNLOADS)

    groups_data = await fetch_json(session, f'https://tcgcsv.com/tcgplayer/{sid}/groups')
    if not groups_data:
        print(f"  No groups returned for category {sid}. Folder not touched.")
        return

    total_products = 0
    for group in groups_data:
        group_id = group.get('groupId')
        if not group_id:
            continue

        set_name = group.get('name') or f"set_{group_id}"
        set_name = "".join(c for c in set_name if c not in '<>:"/\\|?*').strip() or f"set_{group_id}"
        set_dir = os.path.join(save_dir, set_name)

        products = await fetch_json(session, f'https://tcgcsv.com/tcgplayer/{sid}/{group_id}/products')
        total_products += len(products)
        tasks = []
        for item in products:
            item["game_name"] = name
            item["set_name"] = set_name
            tasks.append(download_image(session, item, set_dir, skipped_ids, existing_files, semaphore))
        if tasks:
            await asyncio.gather(*tasks)
            if SHOW_PROGRESS:
                print(f"  {set_name}: {new_dl_count}/{total_products} images so far")

    now = datetime.now().strftime("%Y-%m-%d %H:%M")
    master_coll().update_one(
        {"tcg_display_name": name, "language": "english"},
        {"$set": {"last_run": now}}
    )

    if new_dl_count > 0:
        print(f"\nFinished {name}! Saved {new_dl_count} new images.")
    else:
        print(f"\nNo new cards for {name}. Folder not touched.")


# ==============================================================
# 5. MAIN
# ==============================================================
async def main_async():
    async with aiohttp.ClientSession(headers={'User-Agent': 'CatalogBuilder/1.2.0'}) as session:
        rows = list(master_coll().find(
            {"language": "english", "site_id": {"$nin": [None, "", 0]}}
        ).sort("tcg_display_name", 1))
        global_total = skipped_coll().count_documents({})

        if not rows:
            print("No games found in tcg_master.")
            return

        category_map, categories = await load_category_map(session)
        if not categories:
            print("WARNING: could not load categories from tcgcsv. Aborting.")
            return

        show_menu(rows, global_total)

        if RUN_MODE == "menu":
            print("\nSet RUN_MODE to 'all' or a row number, then re-run this cell.")
            return

        if RUN_MODE == "all":
            for r in rows:
                await process_tcg(session, r, category_map, categories)
            print("\nBatch run complete.")
            return

        try:
            await process_tcg(session, rows[int(RUN_MODE) - 1], category_map, categories)
        except (ValueError, IndexError):
            print(f"Invalid RUN_MODE: {RUN_MODE}. Use 'all', 'menu', or 1-{len(rows)}.")


await main_async()





ENGLISH TCG HARVESTER (MONGODB / JUPYTER EDITION)
Total skipped records: 711
----------------------------------------------------------------------------------------------------
#    TCG CATEGORY              | SITE ID    | LAST RUN           | FOLDER
----------------------------------------------------------------------------------------------------
1    Akora                     | 75         | 2026-09-19 05:33   | Akora
2    Alpha Clash               | 78         | 2026-09-19 05:33   | Alpha Clash
3    Argent Saga               | 61         | 2026-09-19 05:34   | Argent Saga
4    Bakugan                   | 58         | 2026-09-19 05:34   | Bakugan
5    Battle Spirits Saga       | 72         | 2026-09-19 05:34   | Battle Spirits Saga
6    Cardfight Vanguard        | 16         | 2026-09-19 05:39   | Cardfight Vanguard
7    Caster Chronicles         | 37         | 2026-09-19 05:39   | Caster Chronicles
8    Chrono Clash System       | 60         | 2026-09-19 05:39   | Chrono Clash Sy

In [19]:
import asyncio, aiohttp
import config, db

async def probe():
    master = db.get_db()[config.TCG_MASTER_COLLECTION]
    row = master.find_one({"language": "english", "site_id": {"$nin": [None, "", 0]}})
    if not row:
        print("No game with a site_id found in tcg_master.")
        return
    print("Game:", row.get("tcg_display_name"), "| site_id:", row.get("site_id"))
    async with aiohttp.ClientSession(headers={'User-Agent': 'Mozilla/5.0'}) as s:
        async with s.get(f"https://tcgcsv.com/tcgplayer/{row.get('site_id')}/groups", timeout=15) as r:
            print("API status:", r.status)
            print("body:", (await r.text())[:200])

await probe()



Game: Akora | site_id: 75
API status: 200
body: {"totalItems": 10, "success": true, "errors": [], "results": [{"groupId": 23357, "name": "Eternal Echoes [Kickstarter Edition]", "abbreviation": "EEKE", "isSupplemental": false, "publishedOn": "2024-0


In [23]:
import config, db

skipped = db.get_db()[config.SKIPPED_IMAGES_COLLECTION]
master = db.get_db()[config.TCG_MASTER_COLLECTION]

# Total skipped records
print("Total skipped_images records:", skipped.count_documents({}))

# Per-game breakdown for the games you're trying to download
for g in master.find({"language": "english", "site_id": {"$nin": [None, "", 0]}}):
    n = skipped.count_documents({"language": "english", "game_name": g["tcg_display_name"]})
    print(f"  {g['tcg_display_name']:<25} skipped: {n}")


Total skipped_images records: 711
  Akora                     skipped: 0
  Alpha Clash               skipped: 0
  Argent Saga               skipped: 0
  Bakugan                   skipped: 0
  Battle Spirits Saga       skipped: 0
  Cardfight Vanguard        skipped: 38
  Caster Chronicles         skipped: 0
  Chrono Clash System       skipped: 0
  Dice Masters              skipped: 0
  Digimon                   skipped: 169
  DBZ TCG                   skipped: 0
  DBZ Super                 skipped: 0
  DBZ Super Fusion World    skipped: 0
  Dragoborne                skipped: 0
  Elestrals                 skipped: 0
  Final Fantasy             skipped: 0
  Flesh and Blood           skipped: 0
  Force of Will             skipped: 0
  Future Card BuddyFight    skipped: 0
  Gate Ruler                skipped: 0
  Godzilla Card Game        skipped: 0
  Grand Archive             skipped: 60
  Gundam                    skipped: 0
  Hololive                  skipped: 0
  Kryptik                 

In [25]:
import asyncio, aiohttp
import config, db

async def diag():
    master = db.get_db()[config.TCG_MASTER_COLLECTION]
    row = master.find_one({"language": "english", "site_id": {"$nin": [None, "", 0]}})
    sid = row["site_id"]
    print("Game:", row["tcg_display_name"], "| site_id:", sid)

    async with aiohttp.ClientSession(headers={'User-Agent': 'Mozilla/5.0'}) as s:
        async with s.get(f"https://tcgcsv.com/tcgplayer/{sid}/groups", timeout=15) as r:
            groups = (await r.json(content_type=None))["results"]
        print("groups:", len(groups))

        total, with_img = 0, 0
        for g in groups[:3]:
            gid = g["groupId"]
            async with s.get(f"https://tcgcsv.com/tcgplayer/{sid}/{gid}/products", timeout=15) as r:
                prods = (await r.json(content_type=None))["results"]
            total += len(prods)
            with_img += sum(1 for p in prods if p.get("imageUrl"))
            if prods:
                print(f"  group {gid}: {len(prods)} products, {sum(1 for p in prods if p.get('imageUrl'))} with imageUrl")
                print("    first product keys:", list(prods[0].keys()))
        print(f"TOTAL: {total} products, {with_img} with imageUrl")

await diag()



Game: Akora | site_id: 75
groups: 10
  group 23294: 376 products, 376 with imageUrl
    first product keys: ['productId', 'name', 'cleanName', 'imageUrl', 'categoryId', 'groupId', 'url', 'modifiedOn', 'imageCount', 'presaleInfo', 'extendedData']
  group 23300: 349 products, 349 with imageUrl
    first product keys: ['productId', 'name', 'cleanName', 'imageUrl', 'categoryId', 'groupId', 'url', 'modifiedOn', 'imageCount', 'presaleInfo', 'extendedData']
TOTAL: 725 products, 725 with imageUrl


In [34]:
import aiohttp, asyncio

async def check():
    rows = list(master_coll().find(
        {"language": "english", "site_id": {"$nin": [None, "", 0]}}
    ).sort("tcg_display_name", 1))

    async with aiohttp.ClientSession(headers={'User-Agent': 'CatalogBuilder/1.2.0'}) as s:
        for r in rows:
            sid = r.get('site_id')
            url = f'https://tcgcsv.com/tcgplayer/{sid}/groups'
            try:
                async with s.get(url, timeout=15) as resp:
                    data = await resp.json(content_type=None)
                    n = len(data.get('results', []))
                    print(f"{r.get('tcg_display_name',''):<30} sid={sid} -> {n} groups (HTTP {resp.status})")
            except Exception as e:
                print(f"{r.get('tcg_display_name',''):<30} sid={sid} -> ERROR {e}")

await check()



Akora                          sid=75 -> 10 groups (HTTP 200)
Alpha Clash                    sid=78 -> 21 groups (HTTP 200)
Argent Saga                    sid=61 -> 10 groups (HTTP 200)
Bakugan                        sid=58 -> 9 groups (HTTP 200)
Battle Spirits Saga            sid=72 -> 20 groups (HTTP 200)
Cardfight Vanguard             sid=16 -> 272 groups (HTTP 200)
Caster Chronicles              sid=37 -> 12 groups (HTTP 200)
Chrono Clash System            sid=60 -> 5 groups (HTTP 200)
CookieRun Braverse TCG         sid=90 -> 10 groups (HTTP 200)
Cyberpunk TCG                  sid=92 -> 13 groups (HTTP 200)
DBZ Super                      sid=27 -> 107 groups (HTTP 200)
DBZ Super Fusion World         sid=80 -> 52 groups (HTTP 200)
DBZ TCG                        sid=23 -> 9 groups (HTTP 200)
Dice Masters                   sid=18 -> 40 groups (HTTP 200)
Digimon                        sid=63 -> 103 groups (HTTP 200)
Dragoborne                     sid=28 -> 11 groups (HTTP 200)
Elestral

In [36]:
from pprint import pprint
import db

dbs = db.get_db()
print("Database:", dbs.name)

# Show a sample doc from up to 5 collections
for coll_name in dbs.list_collection_names()[:5]:
    coll = dbs[coll_name]
    count = coll.count_documents({})
    sample = coll.find_one()
    print(f"\n=== {coll_name} ({count:,} documents) ===")
    if sample:
        # Trim long string values so output stays readable
        trimmed = {k: (str(v)[:120] + "..." if isinstance(v, str) and len(str(v)) > 120 else v)
                   for k, v in sample.items()}
        pprint(trimmed)
    else:
        print("  (empty collection)")


Database: carddb

=== skipped_images (711 documents) ===
{'_id': ObjectId('6aa93ef91e8242d812b5a8f1'),
 'added_at': '2026-03-17 00:26',
 'game_name': 'Cardfight Vanguard',
 'image_name': '214987',
 'language': 'english'}

=== tcg_folder_harvester (2,165 documents) ===
{'HD': 1,
 '_id': ObjectId('6aacdf6dc86accbbd4914a8a'),
 'confirmed': True,
 'extension': '.jpg',
 'file_size': 62877,
 'filename': '713237.jpg',
 'game': 'Yugioh',
 'group_name': 'Glorious Victors',
 'last_seen': '2026-09-18T02:51:29.801664',
 'modified_time': '2026-09-18T02:51:33.723233',
 'path': 'T:\\Full Card Database\\New Cards\\Yugioh\\Glorious '
         'Victors\\713237.jpg',
 'product_id': '713237',
 'record_type': 'image',
 'status': 'available',
 'updated_at': '2026-09-18T19:14:38.230799'}

=== last_run (18 documents) ===
{'_id': ObjectId('6aa8f2651e8242d8129e8840'),
 'last_run_time': '2026-03-20T00:09:47.696012',
 'script_name': 'jap_cards_main.py',
 'status': 'Failed'}

=== progress (643,030 documents) ===
{

In [37]:
import db
coll = db.get_db()["tcg_folder_harvester"]
deleted = coll.delete_many({})
print(f"Cleared tcg_folder_harvester: {deleted.deleted_count} documents removed.")


Cleared tcg_folder_harvester: 2165 documents removed.
